# 03 DiD Estimation

This notebook estimates baseline regressions, continuous-treatment DiD models, heterogeneity models, and robustness checks.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import pyfixest as pf

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "final_dataset_ai_exposure_controls.csv")

df["post"] = (df["year"] >= 2023).astype(int)
df["ai_post"] = df["ai_exposure"] * df["post"]

# Harmonise control names if needed
df = df.rename(columns={
    "log_Median wage": "log_median_wage",
    "log_Median": "log_median_wage",
    "Median wage": "median_wage"
})

if "log_median_wage" in df.columns:
    df["log_median_wage"] = pd.to_numeric(df["log_median_wage"], errors="coerce")
else:
    print("Warning: log_median_wage not found. Controlled models require this variable.")

df["log_bestand"] = pd.to_numeric(df["log_bestand"], errors="coerce")

print(df.shape)
df.head()

## 1. Baseline specifications: full sample

In [ ]:
m1_relation = pf.feols(
    "log_relation ~ ai_exposure | year",
    data=df,
    vcov={"CRV1": "kldb3"}
)

m2_vakanz = pf.feols(
    "log_vakanz ~ ai_exposure | year",
    data=df,
    vcov={"CRV1": "kldb3"}
)

print(m1_relation.summary())
print(m2_vakanz.summary())

## 2. Continuous-treatment DiD without controls

In [ ]:
did_relation = pf.feols(
    "log_relation ~ ai_exposure:post | kldb3 + year",
    data=df,
    vcov={"CRV1": "kldb3"}
)

did_vakanz = pf.feols(
    "log_vakanz ~ ai_exposure:post | kldb3 + year",
    data=df,
    vcov={"CRV1": "kldb3"}
)

print(did_relation.summary())
print(did_vakanz.summary())

## 3. Continuous-treatment DiD with controls

In [ ]:
control_vars = ["log_bestand", "log_median_wage"]

df_controls = df.dropna(subset=[
    "log_relation", "log_vakanz", "ai_exposure", "post",
    "log_bestand", "log_median_wage", "kldb3", "year"
]).copy()

did_relation_controls = pf.feols(
    "log_relation ~ ai_exposure:post + log_bestand + log_median_wage | kldb3 + year",
    data=df_controls,
    vcov={"CRV1": "kldb3"}
)

did_vakanz_controls = pf.feols(
    "log_vakanz ~ ai_exposure:post + log_bestand + log_median_wage | kldb3 + year",
    data=df_controls,
    vcov={"CRV1": "kldb3"}
)

print(did_relation_controls.summary())
print(did_vakanz_controls.summary())

## 4. Binary treatment robustness: high vs low AI exposure

In [ ]:
median_ai = df_controls["ai_exposure"].median()
df_controls["high_ai"] = (df_controls["ai_exposure"] > median_ai).astype(int)

hetero_relation = pf.feols(
    "log_relation ~ high_ai:post + log_bestand + log_median_wage | kldb3 + year",
    data=df_controls,
    vcov={"CRV1": "kldb3"}
)

hetero_vakanz = pf.feols(
    "log_vakanz ~ high_ai:post + log_bestand + log_median_wage | kldb3 + year",
    data=df_controls,
    vcov={"CRV1": "kldb3"}
)

print(hetero_relation.summary())
print(hetero_vakanz.summary())

## 5. COVID-19 robustness: exclude 2020 and 2021

In [ ]:
df_nocovid = df_controls[~df_controls["year"].isin([2020, 2021])].copy()

did_relation_nocovid = pf.feols(
    "log_relation ~ ai_exposure:post + log_bestand + log_median_wage | kldb3 + year",
    data=df_nocovid,
    vcov={"CRV1": "kldb3"}
)

did_vakanz_nocovid = pf.feols(
    "log_vakanz ~ ai_exposure:post + log_bestand + log_median_wage | kldb3 + year",
    data=df_nocovid,
    vcov={"CRV1": "kldb3"}
)

print(did_relation_nocovid.summary())
print(did_vakanz_nocovid.summary())

## 6. Post-ChatGPT sample robustness: 2023–2025

In [ ]:
df_post = df[df["year"] >= 2023].copy()

fit_post_relation = pf.feols(
    "log_relation ~ ai_exposure | year",
    data=df_post,
    vcov={"CRV1": "kldb3"}
)

fit_post_vakanz = pf.feols(
    "log_vakanz ~ ai_exposure | year",
    data=df_post,
    vcov={"CRV1": "kldb3"}
)

print(fit_post_relation.summary())
print(fit_post_vakanz.summary())

## 7. Export compact coefficient table

In [ ]:
models = {
    "baseline_relation": m1_relation,
    "baseline_vakanz": m2_vakanz,
    "did_relation_controls": did_relation_controls,
    "did_vakanz_controls": did_vakanz_controls,
    "hetero_relation": hetero_relation,
    "hetero_vakanz": hetero_vakanz,
    "post_relation": fit_post_relation,
    "post_vakanz": fit_post_vakanz,
}

rows = []
for model_name, model in models.items():
    tidy = model.tidy().reset_index()
    tidy["model"] = model_name
    rows.append(tidy)

results_table = pd.concat(rows, ignore_index=True)
results_table.to_csv(TABLES / "regression_results_tidy.csv", index=False)
results_table.head()